# 05 病灶自动勾画
在有完整影像和 NVIDIA 显卡的电脑上，直接训练一个二维 U-Net。输入是 T1-FLAIR、T2、DWI 三个通道，输出是背景及标签 1、2、3。

这里采用 Notebook 内的 U-Net 基线，不再依赖 nnU-Net 工程和它的目录配置。每位患者只有选定四层参与训练和评价；其他层不当作无病灶。这项实验验证的是已选层面上的自动勾画，尚不包含自动选层。


## 路径与训练参数


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("分类输入应为每例4层、3通道、224×224。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
cases = read_cases()
SEG_CACHE = COHORT.parent / "segmentation_slices"
OUTPUT = RESULTS / "segmentation"
IMAGE_SIZE = 512
BASE_CHANNELS = 16
EPOCHS = 40
BATCH_SIZE = 2
LEARNING_RATE = 1e-3
PREPARE_CACHE = False
RUN_TRAINING = False
RUN_DEV_PREDICTION = False
RUN_TEST_PREDICTION = False
RESTART = False


## 读取标注层
影像按每个序列的四层分位数归一化。分割单独保留 512×512 的工作尺寸，避免直接用分类缓存中的 224×224 小图训练。原图大小不同时，影像线性缩放、标签最近邻缩放。

`patients_root` 填直接包含各患者文件夹的目录；每例读取 `患者文件夹/配准图像`，患者文件夹名以 MR 编号结尾，例如 `姓名 MR250418095`。


In [ ]:
import re
import nibabel as nib


def image_directories():
    root = path_from_config("patients_root")
    if not root.is_dir():
        raise FileNotFoundError(f"患者总目录不存在：{root}")

    folders = {}
    for patient_dir in sorted(root.iterdir()):
        if not patient_dir.is_dir():
            continue
        match = re.search(r"MR\s*(\d+)$", patient_dir.name.strip(), re.IGNORECASE)
        if not match:
            continue
        directory = patient_dir / "配准图像"
        if not (directory / "t1_flair.nii.gz").is_file():
            continue

        mr = "MR" + match.group(1)
        directory = directory.resolve()
        if mr in folders and folders[mr] != directory:
            raise ValueError(
                f"两个患者文件夹使用了同一个影像号 {mr}：\n"
                f"{folders[mr]}\n{directory}"
            )
        folders[mr] = directory

    if not folders:
        raise FileNotFoundError(
            f"未在 {root} 找到 患者文件夹/配准图像/t1_flair.nii.gz。"
            "请把 patients_root 设为直接包含各患者文件夹的目录。"
        )
    return folders


def read_four_slices(patient, folders=None):
    recorded = [Path(str(patient.get(key, ""))) for key in SEQUENCES + ["manual_mask"]]
    if not all(file.is_file() for file in recorded):
        folders = folders if folders is not None else image_directories()
        if patient.mr not in folders:
            raise FileNotFoundError(f"找不到{patient.mr}的三序列目录。")
        recorded = [folders[patient.mr] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    volumes = [nib.load(str(file)) for file in recorded]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, atol=tolerance, rtol=0) for v in volumes[1:]):
        raise ValueError(f"{patient.mr}三序列与标注的shape/affine不一致。")
    layer = float(patient.max_layer)
    if not np.isfinite(layer) or not layer.is_integer():
        raise ValueError(f"{patient.mr}起始层不是整数。")
    start = int(layer) - 1
    if start < 0 or start + 4 > ref.shape[2]:
        raise ValueError(f"{patient.mr}四层取层越界。")
    labels = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(labels).all() or not set(np.unique(labels)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr}标签必须为0/1/2/3。")
    if np.any(labels[:, :, :start]) or np.any(labels[:, :, start + 4:]):
        raise ValueError(f"{patient.mr}指定四层以外仍有标注，请核对起始层。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, start:start + 4] for v in volumes[:3]]
    if any(not np.isfinite(image).all() for image in images):
        raise ValueError(f"{patient.mr}影像包含无效数值。")
    return images, labels[:, :, start:start + 4].astype(np.uint8)


def normalize_four(volume):
    lo, hi = np.percentile(volume, [1, 99.5])
    return np.clip((volume - lo) / max(float(hi - lo), 1e-6), 0, 1)


## U-Net、损失与训练
编码器逐步提取特征，解码器结合对应层的特征恢复空间细节。损失是交叉熵与前景 Dice 损失之和。

使用与分类相同的患者级五折，每折固定训练上方设定的轮数；不根据时间测试集选模型。开发患者由其留出折模型预测，时间测试患者由五折模型平均概率后预测。每轮保存一次进度，中断后重新运行可从上一轮继续；改模型或训练参数后设置 `RESTART=True`。


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from scipy.ndimage import zoom

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def prepare_segmentation_cache(cases, cache_dir, size=512, overwrite=False):
    cache_dir.mkdir(parents=True, exist_ok=True)
    folders = image_directories()
    for number, patient in enumerate(cases.itertuples(index=False), 1):
        target = cache_dir / f"{patient.mr}.npz"
        if target.exists() and not overwrite:
            continue
        volumes, labels = read_four_slices(pd.Series(patient._asdict()), folders)
        h, w = labels.shape[:2]
        volumes = [normalize_four(v) for v in volumes]
        images = np.stack([np.stack([zoom(v[:, :, s], (size / h, size / w), order=1, prefilter=False)
                                    for v in volumes]) for s in range(4)]).astype(np.float16)
        masks = np.stack([zoom(labels[:, :, s], (size / h, size / w), order=0, prefilter=False) for s in range(4)]).astype(np.uint8)
        np.savez_compressed(target, images=images, labels=masks)
        print(f"分割缓存 {number}/{len(cases)}：{patient.mr}", flush=True)


class SliceDataset(Dataset):
    def __init__(self, cases, cache_dir, image_size):
        images, masks = [], []
        self.records = cases.loc[cases.index.repeat(4)].reset_index(drop=True)
        for mr in cases.mr:
            with np.load(cache_dir / f"{mr}.npz", allow_pickle=False) as data:
                if data["images"].shape != (4, 3, image_size, image_size):
                    raise ValueError("分割缓存尺寸不同，请统一IMAGE_SIZE或重新生成分割缓存。")
                images.append(data["images"])
                masks.append(data["labels"])
        self.images, self.labels = np.concatenate(images), np.concatenate(masks)
        if self.labels.shape != (len(self.images), image_size, image_size) or not np.isin(self.labels, [0, 1, 2, 3]).all():
            raise ValueError("分割标签应为与影像对应的0/1/2/3数组。")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return torch.from_numpy(self.images[index]), torch.from_numpy(self.labels[index])


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, out_channels), nn.ReLU(), nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(4, out_channels), nn.ReLU())

    def forward(self, x):
        return self.net(x)


class UNet2D(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.encoders = nn.ModuleList([DoubleConv(3, base), DoubleConv(base, base * 2),
            DoubleConv(base * 2, base * 4), DoubleConv(base * 4, base * 8)])
        self.bottleneck = DoubleConv(base * 8, base * 16)
        self.up = nn.ModuleList([nn.ConvTranspose2d(base * factor * 2, base * factor, 2, stride=2) for factor in (8, 4, 2, 1)])
        self.decoders = nn.ModuleList([DoubleConv(base * factor * 2, base * factor) for factor in (8, 4, 2, 1)])
        self.output = nn.Conv2d(base, 4, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        for up, decoder, skip in zip(self.up, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([skip, x], dim=1))
        return self.output(x)


def segmentation_loss(logits, labels):
    ce = F.cross_entropy(logits, labels)
    probabilities = logits.float().softmax(dim=1)[:, 1:]
    target = F.one_hot(labels, num_classes=4).permute(0, 3, 1, 2).float()[:, 1:]
    intersection = (probabilities * target).sum(dim=(0, 2, 3))
    dice = (2 * intersection + 1) / (probabilities.sum(dim=(0, 2, 3)) + target.sum(dim=(0, 2, 3)) + 1)
    return ce + 1 - dice.mean()


def load_segmenter(file):
    saved = torch.load(file, map_location="cpu", weights_only=True)
    model = UNet2D(saved["base"]).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), saved


def train_segmentation(cases, cache_dir, output, size=512, base=16, epochs=40, batch_size=2, lr=1e-3, restart=False):
    output.mkdir(parents=True, exist_ok=True)
    dev = cases.query("split == 'development'").reset_index(drop=True)
    if set(dev.development_fold) != {1, 2, 3, 4, 5}:
        raise ValueError("开发患者的折号应为1～5。")
    dataset = SliceDataset(dev, cache_dir, size)
    for fold in range(1, 6):
        final_file, last_file = output / f"fold_{fold}.pt", output / f"fold_{fold}_last.pt"
        if final_file.exists() and not restart:
            print(f"分割fold {fold}已完成，跳过训练。")
            continue
        torch.manual_seed(SEED + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED + fold)
        indices = np.flatnonzero(dataset.records.development_fold.to_numpy() != fold)
        loader = DataLoader(Subset(dataset, indices.tolist()), batch_size=batch_size, shuffle=True, num_workers=0)
        model = UNet2D(base).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
        start, history = 1, []
        if last_file.exists() and not restart:
            saved = torch.load(last_file, map_location="cpu", weights_only=True)
            model.load_state_dict(saved["state_dict"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            start, history = int(saved["epoch"]) + 1, saved["history"]
        for epoch in range(start, epochs + 1):
            model.train()
            total = 0.0
            for x, labels in loader:
                x, labels = x.to(DEVICE, dtype=torch.float32), labels.to(DEVICE, dtype=torch.long)
                if torch.rand(()) < .5:
                    x, labels = x.flip(-1), labels.flip(-1)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                    loss = segmentation_loss(model(x), labels)
                if not torch.isfinite(loss):
                    raise ValueError("分割训练损失出现无效值。")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                total += loss.item() * len(x)
            history.append({"epoch": epoch, "loss": total / len(indices)})
            print(f"seg fold={fold}, epoch={epoch}/{epochs}, loss={history[-1]['loss']:.4f}", flush=True)
            torch.save({"state_dict": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "history": history}, last_file)
        torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "base": base, "image_size": size,
                    "epochs": epochs, "train_mrs": dev.loc[dev.development_fold != fold, "mr"].tolist()}, final_file)
        pd.DataFrame(history).to_csv(output / f"fold_{fold}_history.csv", index=False)
        del model


@torch.no_grad()
def segment_patient(models, images, batch_size=2):
    scores = np.zeros((len(images), 4, *images.shape[-2:]), dtype=np.float32)
    for model in models:
        model.eval()
        for start in range(0, len(images), batch_size):
            x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                probability = model(x).float().softmax(dim=1)
            scores[start:start + len(x)] += probability.cpu().numpy()
    return scores.argmax(axis=1).astype(np.uint8)


def predict_segmentation(cases, cache_dir, output, scope="validation"):
    if scope not in ("validation", "development"):
        raise ValueError("scope只能为development或validation。")
    chosen = cases.loc[cases.split == scope].reset_index(drop=True)
    loaded = [load_segmenter(output / f"fold_{fold}.pt") for fold in range(1, 6)]
    models, metadata = [m for m, _ in loaded], [info for _, info in loaded]
    auto_masks = np.zeros((len(chosen), 4, 3, 224, 224), dtype=np.uint8) if scope == "validation" else None
    rows = []
    prediction_dir = output / f"{scope}_predictions"
    prediction_dir.mkdir(parents=True, exist_ok=True)
    for i, patient in chosen.iterrows():
        with np.load(cache_dir / f"{patient.mr}.npz", allow_pickle=False) as data:
            images, truth = data["images"], data["labels"]
        selected_folds = [int(patient.development_fold) - 1] if scope == "development" else list(range(5))
        if any(patient.mr in metadata[f]["train_mrs"] for f in selected_folds):
            raise ValueError(f"{patient.mr}在用于预测的模型训练集中。")
        if any(images.shape[-1] != metadata[f]["image_size"] for f in selected_folds):
            raise ValueError("预测缓存尺寸与分割模型不同。")
        prediction = segment_patient([models[f] for f in selected_folds], images)
        np.savez_compressed(prediction_dir / f"{patient.mr}.npz", labels=prediction)
        for label in (1, 2, 3):
            a, b = truth == label, prediction == label
            total = int(a.sum() + b.sum())
            rows.append({"mr": patient.mr, "label": label, "dice": float(2 * (a & b).sum() / total) if total else np.nan,
                         "true_pixels": int(a.sum()), "predicted_pixels": int(b.sum())})
        if auto_masks is not None:
            for s in range(4):
                for label in (1, 2, 3):
                    auto_masks[i, s, label - 1] = zoom((prediction[s] == label).astype(np.uint8),
                        (224 / prediction.shape[1], 224 / prediction.shape[2]), order=0, prefilter=False)
        print(f"{scope}分割预测 {i + 1}/{len(chosen)}", flush=True)
    if auto_masks is not None:
        np.savez_compressed(output / "validation_masks.npz", mrs=chosen.mr.to_numpy(dtype=str), masks=auto_masks)
    table = pd.DataFrame(rows)
    table.to_csv(output / f"{scope}_dice.csv", index=False)
    return table.groupby("label").agg(n=("mr", "size"), n_defined=("dice", "count"), mean_dice=("dice", "mean"))


## 准备分割缓存并训练


In [ ]:
if PREPARE_CACHE:
    prepare_segmentation_cache(cases, SEG_CACHE, size=IMAGE_SIZE, overwrite=RESTART)
if RUN_TRAINING:
    train_segmentation(cases, SEG_CACHE, OUTPUT, size=IMAGE_SIZE, base=BASE_CHANNELS,
                       epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, restart=RESTART)


## 预测和评价
Dice 在四个已标注层的工作网格上、按患者和标签计算；真实与预测都没有该标签时记为空值，汇总同时显示有效例数。这里不输出完整病灶体积或毫米距离。

先看开发队列的折外结果，确定训练方案后再生成时间测试结果。测试 mask 会另外转成分类网络需要的 224×224 三个标签通道。


In [ ]:
if RUN_DEV_PREDICTION:
    display(predict_segmentation(cases, SEG_CACHE, OUTPUT, scope="development"))
if RUN_TEST_PREDICTION:
    display(predict_segmentation(cases, SEG_CACHE, OUTPUT, scope="validation"))
if not any([PREPARE_CACHE, RUN_TRAINING, RUN_DEV_PREDICTION, RUN_TEST_PREDICTION]):
    print("依次开启缓存准备、训练、开发预测；方案确定后再开启测试预测。")


## 传到算力机的文件
06 需要 `results/segmentation/validation_masks.npz` 和 `validation_dice.csv`，复制这两个文件即可。前者用于将自动 mask 接入分类网络，后者汇总分割效果。

`fold_1.pt`～`fold_5.pt` 保存分割权重，需要在其他电脑继续分割推理时再复制。`*_last.pt` 是本机续训进度，不必上传到算力机做结果比较。
